In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score



In [ ]:
!pip install catboost
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [ ]:
print(train.shape)
print(test.shape)

(439140, 16)
(188165, 15)


In [ ]:
train = train.dropna(subset=["PitNextLap"])

In [ ]:
target = "PitNextLap"

In [ ]:
def add_features(df):
    df = df.copy()

    df["DegradationRate"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)

    df["TyrePressure"] = df["TyreLife"] / (df["LapNumber"] + 1)

    df["IsLateRace"] = (df["RaceProgress"] > 0.7).astype(int)

    df["PositionPressure"] = df["Position"] * df["Position_Change"]

    df["LapEfficiency"] = df["LapTime (s)"] / (df["LapNumber"] + 1)

    return df

In [ ]:
train = add_features(train)
test = add_features(test)

In [ ]:
def add_aggregations(train, test):
    train = train.copy()
    test = test.copy()

    # Driver-level
    driver_stats = train.groupby("Driver")["PitNextLap"].agg(["mean", "count"])
    driver_stats.columns = ["Driver_PitRate", "Driver_Count"]

    train = train.merge(driver_stats, on="Driver", how="left")
    test = test.merge(driver_stats, on="Driver", how="left")


    # Race-level
    race_stats = train.groupby("Race")["PitNextLap"].mean().reset_index()
    race_stats.columns = ["Race", "Race_PitRate"]

    train = train.merge(race_stats, on="Race", how="left")
    test = test.merge(race_stats, on="Race", how="left")


    # Compound-level
    compound_stats = train.groupby("Compound")["PitNextLap"].mean().reset_index()
    compound_stats.columns = ["Compound", "Compound_PitRate"]

    train = train.merge(compound_stats, on="Compound", how="left")
    test = test.merge(compound_stats, on="Compound", how="left")


    return train, test

In [ ]:
train, test = add_aggregations(train, test)

In [ ]:
print(train.shape)
print(test.shape)

(439140, 25)
(188165, 24)


In [ ]:
features = [col for col in train.columns if col not in ["id", target]]

In [ ]:
X = train[features]
y = train[target]

X_test = test[features]

In [ ]:
groups = train["Race"] + "_" + train["Driver"]

In [ ]:
gkf = GroupKFold(n_splits=5)

scores = []

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        eval_metric="AUC",
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=["Driver", "Compound", "Race"],
        use_best_model=True
    )

    preds = model.predict_proba(X_valid)[:, 1]

    score = roc_auc_score(y_valid, preds)
    print("Fold AUC:", score)

    scores.append(score)


========== FOLD 1 ==========
0:	test: 0.9139935	best: 0.9139935 (0)	total: 1.56s	remaining: 26m
200:	test: 0.9394365	best: 0.9394365 (200)	total: 2m 49s	remaining: 11m 13s
400:	test: 0.9430169	best: 0.9430169 (400)	total: 5m 21s	remaining: 8m
600:	test: 0.9448239	best: 0.9448239 (600)	total: 7m 53s	remaining: 5m 14s
800:	test: 0.9458825	best: 0.9458825 (800)	total: 10m 23s	remaining: 2m 34s
999:	test: 0.9465176	best: 0.9465176 (999)	total: 13m	remaining: 0us

bestTest = 0.9465175957
bestIteration = 999

Fold AUC: 0.9465175957209897

========== FOLD 2 ==========
0:	test: 0.9149872	best: 0.9149872 (0)	total: 681ms	remaining: 11m 20s
200:	test: 0.9406925	best: 0.9406925 (200)	total: 2m 44s	remaining: 10m 55s
400:	test: 0.9442539	best: 0.9442539 (400)	total: 5m 15s	remaining: 7m 51s
600:	test: 0.9460957	best: 0.9460957 (600)	total: 7m 46s	remaining: 5m 9s
800:	test: 0.9471149	best: 0.9471149 (800)	total: 10m 17s	remaining: 2m 33s
999:	test: 0.9477956	best: 0.9477956 (999)	total: 12m 48s	r

In [ ]:
print("\n========== RESULTS ==========")
print("Mean AUC:", np.mean(scores))
print("Std AUC:", np.std(scores))


========== RESULTS ==========
Mean AUC: 0.9473573521646739
Std AUC: 0.0008187805308975706


In [ ]:
# ======================
# Final Model Training
# ======================

final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    eval_metric="AUC",
    verbose=200
)

final_model.fit(
    X, y,
    cat_features=["Driver", "Compound", "Race"]
)

0:	total: 1.46s	remaining: 24m 16s
200:	total: 3m 18s	remaining: 13m 8s
400:	total: 6m 16s	remaining: 9m 22s
600:	total: 9m 11s	remaining: 6m 5s
800:	total: 12m 7s	remaining: 3m
999:	total: 15m 2s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='AUC', iterations=1000, l2_leaf_reg=3, learning_rate=0.03, verbose=200)

In [ ]:
test_preds = final_model.predict_proba(X_test)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "PitNextLap": test_preds
})

submission.to_csv("submission_exp3.csv", index=False)

print("Submission file created:", submission.shape)

Submission file created: (188165, 2)


In [ ]:
print(test.shape)
print(submission.shape)

(188165, 24)
(188165, 2)
